# Preprocesamiento y Modelado Predictivo

[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/10ybtOhXt7YEPw3KWzXKhGNupV2ts3W9h?usp=drive_link)

Análisis del abandono de clientes (Telco Customer Churn).

### 📥 Descarga Automática del Dataset
Ejecuta esta celda para descargar el archivo directamente desde un repositorio público cada vez que se reinicie el entorno de Colab. Esto evitará que tengas que subir el archivo manualmente.

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "WA_Fn-UseC_-Telco-Customer-Churn.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "blastchar/telco-customer-churn",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:", df.head())

# ©️ Preprocesamiento y Modelado Predictivo

In [ ]:
# Importaciones
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [ ]:
# Funciones de Limpieza de Datos (Data Cleaning)

def strip_whitespace(dataframe):
    """Elimina espacios en blanco adicionales al inicio y final de los strings."""
    df_cleaned = dataframe.copy()
    for col in df_cleaned.select_dtypes(include=['object']).columns:
        df_cleaned[col] = df_cleaned[col].astype(str).str.strip()
    return df_cleaned

def fix_data_types(dataframe):
    """Corrige tipos de datos incorrectos, como convertir TotalCharges a numérico."""
    df_cleaned = dataframe.copy()
    if 'TotalCharges' in df_cleaned.columns:
        # Coerce reemplaza los espacios en blanco o textos inválidos con NaN
        df_cleaned['TotalCharges'] = pd.to_numeric(df_cleaned['TotalCharges'], errors='coerce')
    return df_cleaned

def handle_missing_values(dataframe):
    """Trata los valores nulos (NaN) imputando según el tipo de dato."""
    df_cleaned = dataframe.copy()
    for col in df_cleaned.columns:
        if df_cleaned[col].isnull().any():
            if df_cleaned[col].dtype in ['float64', 'int64']:
                # Imputar numéricos con la mediana
                median_value = df_cleaned[col].median()
                df_cleaned[col] = df_cleaned[col].fillna(median_value)
            else:
                # Imputar categóricos con la moda
                mode_value = df_cleaned[col].mode()[0]
                df_cleaned[col] = df_cleaned[col].fillna(mode_value)
    return df_cleaned

def drop_unnecessary_columns(dataframe, columns_to_drop=['customerID']):
    """Elimina columnas que no aportan valor predictivo como IDs."""
    df_cleaned = dataframe.copy()
    existing_cols = [col for col in columns_to_drop if col in df_cleaned.columns]
    return df_cleaned.drop(columns=existing_cols)

def clean_data(dataframe):
    """Aplica todo el flujo de limpieza de datos."""
    df_cleaned = strip_whitespace(dataframe)
    df_cleaned = fix_data_types(df_cleaned)
    df_cleaned = handle_missing_values(df_cleaned)
    df_cleaned = drop_unnecessary_columns(df_cleaned)
    return df_cleaned

# Ejecución de la limpieza sobre nuestro dataset actual
df_clean = clean_data(df)
print("¡Limpieza completada con éxito!")
print("Nuevos tipos de datos:")
print(df_clean.dtypes)

¡Limpieza completada con éxito!
Nuevos tipos de datos:
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                object
dtype: object


# TABLA DE TIPOS DE VARIABLES 🧩🧩

In [ ]:
# Tabla de tipos de Variables (Modificada con mejores prácticas de Preprocesamiento)

def generar_tabla_variables(dataframe):
    variable_info = []
    
    # Clasificación y asignación de tratamientos óptimos basados en el preprocesamiento real
    for col in dataframe.columns:
        unique_vals = dataframe[col].dropna().unique()
        num_unique = len(unique_vals)
        dtype = str(dataframe[col].dtype)
        
        # Definición del tipo de variable y su respectivo tratamiento real aplicado
        if col in ['tenure', 'MonthlyCharges', 'TotalCharges']:
            var_type = 'Numérica Continua/Discreta'
            tratamiento = 'Escalado (StandardScaler)'
        elif col == 'SeniorCitizen':
            var_type = 'Binaria (Numérica)'
            tratamiento = 'Dejala asi nomas'
        elif col == 'Churn':
            var_type = 'Variable Objetivo'
            tratamiento = 'No se escala ni se codifica como una variable mas'
        elif col in ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']:
            var_type = 'Binaria (Nominal)'
            tratamiento = 'Ordinal Encoding'
        elif col == 'Contract':
            var_type = 'Categórica Ordinal'
            tratamiento = 'Ordinal Encoding'
        else:
            var_type = 'Categórica Nominal'
            tratamiento = 'One Hot Encoding'
            
        variable_info.append({
            'Variable': col,
            'Tipo de Dato (Python)': dtype,
            'Tipo de Variable': var_type,
            'Valores Únicos': num_unique,
            'Ejemplos de Valores': list(unique_vals[:3]),
            'Tratamiento Real': tratamiento  
        })
        
    return pd.DataFrame(variable_info)

tabla_variables = generar_tabla_variables(df_clean)
tabla_variables

,Variable,Tipo de Dato (Python),Tipo de Variable,Valores Únicos,Ejemplos de Valores,Tratamiento Real
0,gender,object,Binaria (Nominal),2,"[Female, Male]",Ordinal Encoding
1,SeniorCitizen,int64,Binaria (Numérica),2,"[0, 1]",Dejala asi nomas
2,Partner,object,Binaria (Nominal),2,"[Yes, No]",Ordinal Encoding
3,Dependents,object,Binaria (Nominal),2,"[No, Yes]",Ordinal Encoding
4,tenure,int64,Numérica Continua/Discreta,73,"[1, 34, 2]",Escalado (StandardScaler)
5,PhoneService,object,Binaria (Nominal),2,"[No, Yes]",Ordinal Encoding
6,MultipleLines,object,Categórica Nominal,3,"[No phone service, No, Yes]",One Hot Encoding
7,InternetService,object,Categórica Nominal,3,"[DSL, Fiber optic, No]",One Hot Encoding
8,OnlineSecurity,object,Categórica Nominal,3,"[No, Yes, No internet service]",One Hot Encoding
9,OnlineBackup,object,Categórica Nominal,3,"[Yes, No, No internet service]",One Hot Encoding


# 🅱️ Análisis Exploratorio de Datos (EDA) Categórico y Visualización
### 1. Frecuencias Absolutas, Relativas y Tablas de Contingencia

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Cálculo de Frecuencias Absolutas y Relativas
variables_categoricas = ['Churn', 'Contract', 'PaymentMethod']

print("Frecuencias Absolutas y relativas")
for col in variables_categoricas:
    freq_abs = df_clean[col].value_counts()
    freq_rel = (df_clean[col].value_counts(normalize=True) * 100).round(2)
    
    tabla_freq = pd.DataFrame({
        'Frecuencia Absoluta (Clientes)': freq_abs,
        'Frecuencia Relativa (%)': freq_rel
    })
    print(f"\n Distribución de la variable: {col}")
    display(tabla_freq)

# 2. Tablas de Contingencia respecto a la tasa de cancelación (Churn)
print("Tablas de contigencia")
variables_cruce = ['Contract', 'PaymentMethod', 'InternetService', 'SeniorCitizen']

for col in variables_cruce:
    # Conteo absoluto con totales
    ct_abs = pd.crosstab(df_clean[col], df_clean['Churn'], margins=True, margins_name='Total')
    # Porcentaje por fila (muestra la tasa de abandono de cada categoría)
    ct_rel = (pd.crosstab(df_clean[col], df_clean['Churn'], normalize='index') * 100).round(2)
    ct_rel.columns = ['Retenidos - No (%)', 'Tasa de Churn - Yes (%)']
    
    tabla_contingencia = pd.concat([ct_abs, ct_rel], axis=1)
    print(f"\n Cruce de {col} vs Churn:")
    display(tabla_contingencia)

### 2. Gráficos Comparativos de Cancelación (Churn)

In [ ]:
# Configuración estética de Seaborn
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico 1: Tasa de cancelación según Tipo de Contrato
ax1 = sns.countplot(
    data=df_clean,
    x='Contract',
    hue='Churn',
    palette=['#2ecc71', '#e74c3c'],
    order=['Month-to-month', 'One year', 'Two year'],
    ax=axes[0]
)
axes[0].set_title('Cancelación (Churn) según Tipo de Contrato', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Tipo de Contrato', fontsize=11)
axes[0].set_ylabel('Cantidad de Clientes', fontsize=11)
axes[0].legend(title='¿Canceló?', labels=['No (Continúa)', 'Sí (Churn)'])

for container in ax1.containers:
    ax1.bar_label(container, fontsize=10)

# Gráfico 2: Tasa de cancelación según Modalidad de Pago
ax2 = sns.countplot(
    data=df_clean,
    x='PaymentMethod',
    hue='Churn',
    palette=['#2ecc71', '#e74c3c'],
    ax=axes[1]
)
axes[1].set_title('Cancelación (Churn) según Modalidad de Pago', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Modalidad de Pago', fontsize=11)
axes[1].set_ylabel('Cantidad de Clientes', fontsize=11)
axes[1].tick_params(axis='x', rotation=15)
axes[1].legend(title='¿Canceló?', labels=['No (Continúa)', 'Sí (Churn)'])

for container in ax2.containers:
    ax2.bar_label(container, fontsize=10)

plt.tight_layout()
plt.show()

#### 💡 Interpretación Analítica Orientada al Impacto en el Negocio

1. **Análisis por Tipo de Contrato (`Contract`):**
   * **Hallazgo Estadístico:** Aunque la tasa general de cancelación de la empresa es del **26.54%** (1.869 bajas sobre 7.043 clientes), el abandono está fuertemente concentrado en los contratos mensuales (`Month-to-month`), los cuales representan el **55.02%** de la cartera total y sufren una tasa de cancelación crítica del **42.71%** (1.655 bajas). Por el contrario, en los contratos de 1 año (`One year`) la baja cae al **11.27%**, y en los de 2 años (`Two year`) es de apenas el **2.83%**.
   * **Impacto en el Negocio:** La falta de compromiso contractual a largo plazo es el principal impulsor de pérdida de ingresos recurrentes. Se recomienda diseñar incentivos comerciales (descuentos en el abono mensual o meses bonificados) para migrar a los clientes de modalidad mes a mes hacia contratos anuales.

2. **Análisis por Modalidad de Pago (`PaymentMethod`):**
   * **Hallazgo Estadístico:** Los clientes que abonan mediante cheque electrónico (`Electronic check`, el **33.58%** del total) registran una tasa de abandono sumamente elevada del **45.29%** (1.071 bajas). En contraste, quienes utilizan medios de pago automáticos presentan tasas de fuga casi tres veces menores: **16.71%** en transferencia bancaria automática y **15.24%** en tarjeta de crédito automática.
   * **Impacto en el Negocio:** Los métodos de pago manuales generan fricción mes a mes y exponen al cliente a reconsiderar la continuidad del servicio en cada vencimiento. La estrategia comercial debe enfocarse en bonificar la adhesión al débito automático con tarjeta o cuenta bancaria para fidelizar la cartera.

# Preprocesamiento de Datos para Modelado 🛠️

In [ ]:
# Definicion de que columnas tendra cada tratamiento
columnas_numericas = ['tenure', 'MonthlyCharges', 'TotalCharges']
columnas_binarias = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
columnas_ordinales = ['Contract']
columnas_nominales = ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 
                'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaymentMethod']

# Definicion del orden de contrato (0-2) de menor a mayor
ordenCorrecto_contrato = ['Month-to-month', 'One year', 'Two year']


# Preprocesador utilizando ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), columnas_numericas),
        ('bin_ord', OrdinalEncoder(), columnas_binarias),
        ('contract_ord', OrdinalEncoder(categories=[ordenCorrecto_contrato]), columnas_ordinales),
        ('nom_ohe', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), columnas_nominales)
    ],
    remainder='passthrough' # Para SeniorCitizen sin cambios es 0 y 1 XD
)

# Variables independientes (X) y variable objetivo (y)
X = df_clean.drop(columns=['Churn'])
y = LabelEncoder().fit_transform(df_clean['Churn'])

# Divicion entre Train y Test antes de transformar
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Ajuste del preprocesador con los datos de entrenamiento y transforme de ambos
X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

# Convercion a DataFrame
encoded_cols = (
    columnas_numericas +
    columnas_binarias +
    columnas_ordinales +
    list(preprocessor.named_transformers_['nom_ohe'].get_feature_names_out(columnas_nominales)) +
    ['SeniorCitizen']
)

X_train_df = pd.DataFrame(X_train_prepared, columns=encoded_cols)
display(X_train_df.head())

,tenure,MonthlyCharges,TotalCharges,gender,Partner,Dependents,PhoneService,PaperlessBilling,Contract,MultipleLines_No phone service,...,TechSupport_No internet service,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,SeniorCitizen
0,0.102371,-0.521976,-0.263289,1.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
1,-0.711743,0.337478,-0.504814,1.0,1.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,-0.793155,-0.809013,-0.751213,1.0,1.0,1.0,0.0,0.0,2.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,-0.263980,0.284384,-0.173699,0.0,1.0,0.0,1.0,1.0,2.0,0.0,...,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0
4,-1.281624,-0.676279,-0.990851,1.0,1.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


# VERIFICACION DE RESULTADOS 📎

In [ ]:
prep_df = pd.DataFrame(X_train_prepared, columns=preprocessor.get_feature_names_out())

print('1. ¿Todo numérico?              ', prep_df.dtypes.nunique() == 1 and prep_df.dtypes.iloc[0] == 'float64')
print('2. Valores nulos:               ', int(np.isnan(X_train_prepared).sum()))
print('3. ¿Mismas columnas train/test? ', X_train_prepared.shape[1] == X_test_prepared.shape[1])
print('4. Proporción y_train:          ', pd.Series(y_train).value_counts(normalize=True).round(3).to_dict())
print('   Proporción y_test:           ', pd.Series(y_test).value_counts(normalize=True).round(3).to_dict())
print('5. Numéricas de train → media:  ', X_train_prepared[:, :3].mean(axis=0).round(2),
      ' desvío:', X_train_prepared[:, :3].std(axis=0).round(2))

# **Predicción Numérica: Modelos de Regresión**📈📉

In [ ]:
# Inicializacion del modelo
modelo = LinearRegression()

# Entrenamiento para estimar SOLO una variable continua (TotalCharges)

# Serea comparada con tenure (meses)
X_s = df_clean[['tenure']]
y_s = df_clean['TotalCharges']

# 2. División en conjuntos de entrenamiento y prueba (80/20)
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_s, y_s, test_size=0.2, random_state=42
)

# Entrenamiento del modelo utilizando los datos preparados
modelo.fit(X_train_s, y_train_s)

# Consulta de datos aprendidos
# Por cada mes adicional que un cliente permanece en la empresa, sus cargos totales acumulados aumentan $76.59
print("Pendiente del primer coeficiente:", modelo.coef_[0])
print("Intercepto:", modelo.intercept_)

# Prediccion utilizando los datos de test preparados
predicciones = modelo.predict(X_test_s)
print("Predicciones (primeras 5):", predicciones[:5])

# Comparacion con respecto a los datos originales
comparacion = pd.DataFrame({
    'Valor Real': y_test_s,
    'Prediccion': predicciones
})

comparacion = comparacion.head(10)

# Para un cliente con 1 mes de servicio (Fila 185): Su cargo real fue de $24.80, pero el modelo predijo -$113.43. Para clientes nuevos el modelo simple no es tan exacto
# Para un cliente con 41 meses de servicio (Fila 2715): Su cargo real fue de $996.45 y el modelo predijo $2,950.34.
display(comparacion.sort_values('Prediccion'))

# Evaluaciones simples 📋

In [ ]:
# MAE: Error absoluto Medio
mae = mean_absolute_error(y_test_s, predicciones)
print("Error Absoluto Medio (MAE):", mae) # -> Se equivoca $892.0415772496411 en promedio

# MSE: Error cuadrático medio
mse = mean_squared_error(y_test_s, predicciones)
print("Error Cuadrático Medio (MSE):", mse) # -> 1717993.9247736863 La empresa ya puede ir pensado en cerrar XD

# R**2: Coeficiente de determinacion
r2 = r2_score(y_test_s, predicciones)
print("Coeficiente de Determinación (R**2):", r2) # -> 0.6695741445638115

metricas = pd.DataFrame({
    "Métrica": ["MAE", "MSE", "R²"],
    "Resultado": [mae, mse, r2]
})

display(metricas)


# VISUALIZACION DE LOS RESULTADOS FINALES 📊

In [ ]:
plt.figure(figsize=(10, 6))

# Datos reales de entrenamiento (en azul)
plt.scatter(
    X_train_s, y_train_s, color='blue', label='Datos de Entrenamiento (80%)', alpha=0.4
)

# Datos reales de prueba reservados (en verde)
plt.scatter(
    X_test_s, y_test_s, color='green', label='Datos de Prueba (20%)', alpha=0.4
)

# Recta de regresión aprendida por el modelo
 # La linea de regresion estaba cortada en partes, esta fue la mejor solucion
X_valores_ordenados = pd.DataFrame(np.sort(X_s.values, axis=0), columns=X_s.columns)
predicciones_ordenadas = modelo.predict(X_valores_ordenados)
plt.plot(
    X_valores_ordenados, predicciones_ordenadas, color='red', linewidth=3, label='Recta de Regresión (Modelo)'
)

# Configuración de etiquetas y títulos correctos
plt.xlabel("Tenure / Permanencia (Meses)", fontsize=12)
plt.ylabel("Cargos Totales / Total Charges ($)", fontsize=12)
plt.title("Regresión Lineal Simple: Permanencia vs Cargos Totales", fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
# FUTURAS PREDICCIONES

# DataFrame con los meses de permanencia
nuevo_cliente = pd.DataFrame({
    'tenure': [24] # Ejemplo: Estimar cargos para un cliente con 2 años de servicio
})

# predicciOn con EL modelo entrenado
cargos_estimados = modelo.predict(nuevo_cliente)

print("--- Predicción Individual ---")
print(f"Permanencia ingresada: {nuevo_cliente['tenure'][0]} meses")
print(f"Cargos Totales estimados: USD {cargos_estimados[0]:,.2f}\n")

# predicciones mUltiples
varios_clientes = pd.DataFrame({
    'tenure': [1, 12, 36, 72] # 1 mes, 1 año, 3 años y 6 años
})

varios_clientes['Cargos Totales Estimados'] = modelo.predict(varios_clientes)
display(varios_clientes)